In [ ]:
from phytclust.main import PhytClust
from phytclust import plot_tree, plot_multiple_clusters
from phytclust import pairwise_distances
import io
import os
from Bio import Phylo
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# with newick string
# tree_string = (
# "(((A:5, B:3)C1:6, (C:3, D:7)D1:4)A13:22, (((E:7, F:13)E12:5, G:6)B23:10, H:60):35):0;"
# )
# tree = Phylo.read(io.StringIO(tree_string), "newick")\

# With newick file
tree_path = "/home/ganesank/project/phytclust/Data/ar53_r220.tree"
tree = Phylo.read(tree_path, "newick", rooted=True)

In [ ]:
clust_obj = PhytClust(tree, should_plot_scores=True, num_peaks=10, resolution_on=True)

In [ ]:
clust_obj.best_peaks

In [ ]:
clust_obj.best_peaks[1]
from collections import Counter
from Bio.Phylo.BaseTree import Clade

print(Counter(clust_obj.clusters[7].values()))

In [ ]:
clust_obj_clusters = [
    clust_obj.clusters[clust_obj.best_peaks[1] - 1],
    clust_obj.clusters[clust_obj.best_peaks[2] - 1],
    # clust_obj.clusters[8],
]
comparison_ids = ["1", "2"]
rows = []

for clade_dict, comparison_id in zip(clust_obj_clusters, comparison_ids):
    for clade, cluster_id in clade_dict.items():
        leaf_name = clade.name
        rows.append([leaf_name, comparison_id, cluster_id])

df_clusters = pd.DataFrame(rows, columns=["leaf_name", "comparison_IDs", "cluster_ID"])

In [ ]:
import os
import logging
import matplotlib.collections as mpcollections
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import logging
from collections import defaultdict
from typing import Any, Callable, Dict, List, Optional, Tuple, Union


def _get_y_positions(
    tree: Any, adjust: bool = False, outgroup: str = "outgroup"
) -> Dict[Any, float]:
    """Create a mapping of each clade to its vertical position.
        Dict of {clade: y-coord}.
    Coordinates are negative, and integers for tips.
    """
    maxheight = tree.count_terminals()
    heights = {
        tip: maxheight - 1 - i
        for i, tip in enumerate(
            reversed(
                [
                    x
                    for x in tree.get_terminals()
                    if outgroup is None or x.name != outgroup
                ]
            )
        )
    }
    if outgroup is not None:
        normal_clades = list(tree.find_clades(outgroup))
        if not normal_clades:
            raise PlotError(f"Normal clade {outgroup} not found in tree")
        heights[normal_clades[0]] = maxheight

    # Internal nodes: place at midpoint of children
    def calc_row(clade: Any) -> None:
        for subclade in clade:
            if subclade not in heights:
                calc_row(subclade)
        # Closure over heights
        heights[clade] = (heights[clade.clades[0]] + heights[clade.clades[-1]]) / 2.0

    if tree.root.clades:
        calc_row(tree.root)

    if adjust:
        pos = pd.DataFrame(
            [(clade, val) for clade, val in heights.items()], columns=["clade", "pos"]
        ).sort_values("pos")
        pos["newpos"] = 0
        count = 0
        for i in pos.index:
            if pos.loc[i, "clade"] != tree.root:
                count += 1
            pos.loc[i, "newpos"] = count

        pos.set_index("clade", inplace=True)
        heights = pos.to_dict()["newpos"]

    return heights


logger = logging.getLogger(__name__)


def plot_multiple_clusters(
    input_df: pd.DataFrame,
    final_tree: Optional[Any] = None,
    y_posns: Optional[Dict[str, int]] = None,
    cmax: Optional[int] = None,
    tree_width_ratio: float = 1,
    cbar_width_ratio: float = 0.05,
    figsize: Tuple[int, int] = (6, 10),
    tree_marker_size: int = 1,
    show_internal_nodes: bool = False,
    title: str = "",
    tree_label_func: Optional[callable] = None,
    cmap: str = "tab20b",
    outgroup: str = "diploid",
    fixed_x_range: Tuple[int, int] = (0, 5000),
) -> plt.Figure:

    # cmax = cmax or np.max(input_df["cluster_ID"].values.astype(int))
    # print(f"cmax: {cmax}")
    sample_labels = input_df["leaf_name"].unique().tolist()
    required_columns = {"leaf_name", "comparison_IDs", "cluster_ID"}
    if not required_columns.issubset(input_df.columns):
        raise ValueError(f"Input DataFrame must contain columns: {required_columns}")
        print(input_df.columns)

    data_matrix = input_df.pivot(
        index="comparison_IDs", columns="leaf_name", values="cluster_ID"
    )

    data_matrix = data_matrix.reindex(columns=sample_labels).fillna(0)
    normalized_matrix = data_matrix.div(data_matrix.max(axis=1), axis=0).fillna(0)
    Z = normalized_matrix.T.values
    color_norm = mcolors.Normalize(vmin=0, vmax=1)

    if final_tree is None:
        fig, ax = plt.subplots(
            figsize=figsize,
            ncols=1,
            sharey=False,
            gridspec_kw={"width_ratios": [1]},
        )
        if not show_internal_nodes:
            logger.warning(
                'No tree provided, so "show_internal_nodes=False" is ignored'
            )
        y_posns = y_posns or {s: i for i, s in enumerate(sample_labels)}
        ax.set_title(
            title,
            x=0,
            y=1,
            ha="left",
            va="bottom",
            pad=20,
            fontweight="bold",
            fontsize=16,
            zorder=10,
        )
    else:
        fig, axs = plt.subplots(
            figsize=(figsize),
            ncols=2,
            sharey=False,
            gridspec_kw={"width_ratios": [tree_width_ratio*0.2, 0.15], "wspace": 0.05},
        )
        tree_ax, ax = axs

        y_posns = {
            k.name: v
            for k, v in _get_y_positions(
                final_tree, adjust=show_internal_nodes, outgroup=outgroup
            ).items()
        }
        plot_tree(
            final_tree,
            ax=tree_ax,
            outgroup=outgroup,
            label_func=tree_label_func or (lambda x: ""),
            hide_internal_nodes=True,
            show_branch_lengths=False,
            show_events=False,
            line_width=0.5,
            marker_size=tree_marker_size,
            title=title,
            label_colors=None,
        )
        tree_ax.set_axis_off()
        # fig.set_constrained_layout_pads(w_pad=0, h_pad=0, hspace=0.0, wspace=150)

    n_samples = len(sample_labels)
    n_comparisons = data_matrix.shape[0]
    x_pos = np.linspace(fixed_x_range[0], fixed_x_range[1], n_comparisons + 1)
    y_pos = np.arange(n_samples + 1) + 0.5

    # color_norm = mcolors.Normalize(vmin=0, vmax=cmax)
    im = ax.pcolormesh(x_pos, y_pos, Z, cmap=cmap, norm=color_norm)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_ylim(n_samples + 0.5, 0.5)

    return fig


fig = plot_multiple_clusters(
    input_df=df_clusters, final_tree=tree, outgroup=None, tree_marker_size=1
)

get mrcas

In [ ]:
from Bio.Phylo.BaseTree import Tree
from collections import defaultdict

def build_parent_map(tree):
    parent_map = {}
    for clade in tree.find_clades(order='level'):
        for child in clade:
            parent_map[child] = clade
    return parent_map

def calculate_mrca_names(tree, cluster_assignments):
    """
    Calculate the MRCA names for each cluster and return a data structure
    with cluster numbers and corresponding MRCA names.
    """
    # Build a parent map for upward traversal (if needed for other purposes)
    parent_map = build_parent_map(tree)

    # Group clade names by cluster
    clusters = defaultdict(list)
    for clade, cluster in cluster_assignments.items():
        if isinstance(clade, str):
            clusters[cluster].append(clade)  # Use clade names directly
        else:
            clusters[cluster].append(clade.name)  # Use clade names from objects

    # Print clade names from the tree for debugging
    tree_clade_names = {clade.name for clade in tree.find_clades() if clade.name}
    # print("Clade names in the tree:", tree_clade_names)

    # Store cluster and MRCA results
    mrca_results = []

    for cluster, member_names in clusters.items():
        # Filter out member names not in the tree
        member_names_in_tree = [name for name in member_names if name in tree_clade_names]
        if not member_names_in_tree:
            print(f"No valid members in tree for cluster {cluster}")
            continue

        # Find MRCA for the cluster using names
        try:
            mrca = tree.common_ancestor(member_names_in_tree)
            mrca_name = mrca.name if mrca.name else f"Unnamed_MRCA_{cluster}"
            mrca_results.append({"cluster": cluster, "mrca_name": mrca_name})
        except ValueError as e:
            print(f"Error finding MRCA for cluster {cluster}: {e}")
            continue

    return mrca_results

mrca_data = calculate_mrca_names(tree, clust_obj.clusters[63])

# Output
for entry in mrca_data:
    print(f"Cluster: {entry['cluster']}, MRCA Name: {entry['mrca_name']}")

# archaea file

In [ ]:
tree_path = "/home/ganesank/project/phytclust/Data/ar53_r220.tree"
tree = Phylo.read(tree_path, "newick", rooted=True)

In [ ]:
import pandas as pd

# Modify this to the path of your input file.
input_file = "/home/ganesank/project/phytclust/Data/ar53_taxonomy.tsv"


def parse_archaea_line(line):
    # Example line:
    # RS_GCF_028743435.1    d__Archaea;p__Methanobacteriota;c__Methanobacteria;o__Methanobacteriales;f__Methanobacteriaceae;g__Methanobrevibacter_A;s__Methanobrevibacter_A smithii

    parts = line.strip().split("\t")
    if len(parts) < 2:
        return None

    taxa = parts[0].strip()
    taxonomy_str = parts[1].strip()

    # Only process lines that clearly have a taxonomy starting with d__Archaea
    if not taxonomy_str.startswith("d__Archaea;"):
        return None

    # Split by ';'
    ranks = taxonomy_str.split(";")

    tax_dict = {}
    for r in ranks:
        if "__" in r:
            rank_code, name = r.split("__", 1)
            tax_dict[rank_code] = name.strip()

    # Extract ranks of interest
    phylum = tax_dict.get("p", "")
    order = tax_dict.get("o", "")
    family = tax_dict.get("f", "")
    genus = tax_dict.get("g", "")
    species = tax_dict.get("s", "")

    return [taxa, phylum, order, family, genus, species]


# We'll store rows in this list
rows = []

with open(input_file, "r") as f:
    for line in f:
        parsed = parse_archaea_line(line)
        if parsed:
            # Archaea line
            rows.append(parsed)
        else:
            # Not an archaea line; check if it's already in the correct format
            cols = line.strip().split("\t")
            if len(cols) >= 6:
                # Already in desired format
                rows.append(cols)
            # Else: skip lines that don't match either pattern

# Create a DataFrame
df = pd.DataFrame(
    rows, columns=["taxa", "phylum", "order", "family", "genus", "species"]
)

# Display the resulting DataFrame
df

In [ ]:
from Bio.Phylo.BaseTree import Clade


phylum_codes, phylum_unique = pd.factorize(df["phylum"])
df["cluster_id"] = phylum_codes
taxa_to_cluster = dict(zip(df["taxa"], df["cluster_id"]))
print(taxa_to_cluster)
tree_clades = [
    Clade(branch_length=terminal.branch_length, name=terminal.name)
    for terminal in tree.get_terminals()
]

clusters_high = {
    clade: taxa_to_cluster.get(clade.name, None)
    for clade in tree_clades
    if clade.name in taxa_to_cluster
}

# Print the clusters_high dictionary
print(clusters_high)

In [ ]:
clust_obj_clusters = [
    clust_obj.clusters[18],
    # clusters_high,
    # clust_obj.clusters[8],
]
comparison_ids = ["1", "2"]
rows = []

for clade_dict, comparison_id in zip(clust_obj_clusters, comparison_ids):
    for clade, cluster_id in clade_dict.items():
        leaf_name = clade.name
        rows.append([leaf_name, comparison_id, cluster_id])

df_clusters = pd.DataFrame(rows, columns=["leaf_name", "comparison_IDs", "cluster_ID"])

In [ ]:
fig = plot_multiple_clusters(
    input_df=df_clusters, final_tree=clust_obj.tree, outgroup=None, tree_marker_size=1
)

In [ ]:
help(plot_multiple_clusters)